<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/textbook_docai_extraction/Textbook_docai_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Textbook → Document AI OCR → target-word sentences (v2)
**PDF → 15-page chunks → Document AI OCR → pages (visual order + confidence) → cleaning → sentence units with page numbers → target-word extraction**. The extraction rules and output columns are the same as the corpus notebook.

### 1. Install

In [2]:
!pip install -q --upgrade google-cloud-documentai pypdf openpyxl

### 2. Settings
* `START = 10`: the first 10 PDF pages are front matter and aren't OCR'd. `book_page = pdf_page - START`.
* `MIN_CONF = 0.85`: pages whose mean **token** confidence is below this are excluded and logged. The threshold comes from SiPaKosa (CHiPSAL 2026).
* `MIN_CHARS, MAX_CHARS, MIN_WORDS, MIN_SI = 20, 400, 4, 0.8`: the sentence filters recorded in the project plan (§4.9).
* outputs stay in `Research-WSD-New/textbooks/Ext_Final/sinhala_lit_g10_11_docai/`.

In [3]:
import os, re, io, json, glob, bisect, random, unicodedata, datetime
from collections import Counter, defaultdict
import pandas as pd
from google.colab import auth, drive
drive.mount("/content/drive")

BASE     = "/content/drive/MyDrive/Research-WSD-New"

PROJECT_ID   = "sinhala-wsd-ocr"
LOCATION     = "us"
PROCESSOR_ID = "7901bb5eb670c044"

BOOK = {
    "book_id":   "sinhala_lit_g10_11",
    "pdf":       f"{BASE}/textbooks/sinhala lit g-10-11.pdf",
    "title":     "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
    "publisher": "Educational Publications Department",
    "grade":     "10-11", "year": "2020", "edition": "7th printing (first printed 2014)",
    "isbn":      "978-955-25-0381-8",
    "url":       "http://edupub.gov.lk/Administrator/Sinhala/10/sin%20lit%20g-10-11/sinhala%20lit%20g-10-11.pdf",
}
# skip front matter and OCR page limit 15
START, CHUNK = 10, 15
MIN_CONF = 0.85
MIN_CHARS, MAX_CHARS, MIN_WORDS, MIN_SI = 20, 400, 4, 0.8

OUT       = f'{BASE}/textbooks/Ext_Final/{BOOK["book_id"]}_docai'
CHUNK_DIR = f"{OUT}/chunks"      # 15-page PDFs made in block 4
RAW       = f"{OUT}/docai_raw"   # one saved OCR response per chunk
for d in (CHUNK_DIR, RAW): os.makedirs(d, exist_ok=True)
print("PDF:", "OK" if os.path.exists(BOOK["pdf"]) else "MISSING", "| out:", OUT)

Mounted at /content/drive
PDF: OK | out: /content/drive/MyDrive/Research-WSD-New/textbooks/Ext_Final/sinhala_lit_g10_11_docai


### 3. Log & Connect

In [4]:
auth.authenticate_user()
from google.api_core.client_options import ClientOptions
from google.cloud import documentai

client = documentai.DocumentProcessorServiceClient(client_options=ClientOptions(
    api_endpoint=f"{LOCATION}-documentai.googleapis.com",
    quota_project_id=PROJECT_ID))

proc = client.get_processor(name=client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID))
assert proc.type_ == "OCR_PROCESSOR", f"wrong processor type: {proc.type_}"
assert proc.state.name == "ENABLED", f"processor is {proc.state.name}"

NAME = proc.default_processor_version
VERSION_USED = NAME.split("/")[-1]
ver = client.get_processor_version(name=NAME)
print("processor:", proc.display_name, "| type:", proc.type_, "| version:", VERSION_USED, f"({ver.display_name})")

processor: sinhala-ocr | type: OCR_PROCESSOR | version: pretrained-ocr-v2.1-2024-08-07 (Google Stable)
